<a href="https://colab.research.google.com/github/vishwaShetti1/New/blob/main/data_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# DATA PIPELINE PROJECT
# PART 1 - SCRAPE BOOK DATA FROM books.toscrape.com
# ============================================================

# ------------------------------------------------------------
# Import Required Libraries
# ------------------------------------------------------------

import requests                  # Used to send HTTP requests
from bs4 import BeautifulSoup    # Used to extract HTML data
import pandas as pd              # Used to store data in DataFrame

# ------------------------------------------------------------
# Website URL
# ------------------------------------------------------------

BASE_URL = "https://books.toscrape.com/catalogue/page-{}.html"

# ------------------------------------------------------------
# Create Empty List
# ------------------------------------------------------------

# Every scraped book will be stored in this list
books = []

# ------------------------------------------------------------
# Rating Conversion Dictionary
# ------------------------------------------------------------

# The website stores ratings as words.
# We will later convert them into numbers.

rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

# ------------------------------------------------------------
# Scrape First 5 Pages
# ------------------------------------------------------------

# Each page contains 20 books.
# Therefore, 5 pages = 100 books.

for page in range(1, 6):

    print(f"\nScraping Page {page}...")

    # Create page URL
    url = BASE_URL.format(page)

    # Send request
    response = requests.get(url)

    # Check whether request is successful
    if response.status_code != 200:
        print("Unable to access page:", page)
        continue

    # Convert HTML into BeautifulSoup object
    soup = BeautifulSoup(response.text, "html.parser")

    # Find all books available on the page
    all_books = soup.find_all("article", class_="product_pod")

    # --------------------------------------------------------
    # Loop through every book
    # --------------------------------------------------------

    for book in all_books:

        # ------------------------------
        # Book Title
        # ------------------------------

        title = book.h3.a["title"]

        # ------------------------------
        # Book Price
        # ------------------------------

        price = book.find(
            "p",
            class_="price_color"
        ).text.strip()

        # ------------------------------
        # Book Rating
        # ------------------------------

        rating_text = book.find(
            "p",
            class_="star-rating"
        )["class"][1]

        # ------------------------------
        # Availability
        # ------------------------------

        availability = book.find(
            "p",
            class_="instock availability"
        ).get_text(strip=True)

        # ----------------------------------------------------
        # Category
        # ----------------------------------------------------
        # Open the product page to get the category

        book_link = book.h3.a["href"]

        # Convert relative URL into complete URL
        book_url = (
            "https://books.toscrape.com/catalogue/"
            + book_link.replace("../", "")
        )

        # Open product page
        book_response = requests.get(book_url)

        if book_response.status_code == 200:

            book_soup = BeautifulSoup(
                book_response.text,
                "html.parser"
            )

            breadcrumb = book_soup.find(
                "ul",
                class_="breadcrumb"
            )

            category = breadcrumb.find_all("li")[2].text.strip()

        else:

            category = "Unknown"

        # ----------------------------------------------------
        # Save Book Information
        # ----------------------------------------------------

        books.append({

            "title": title,

            "price": price,

            "star_rating": rating_text,

            "availability": availability,

            "category": category

        })

# ------------------------------------------------------------
# Convert List into DataFrame
# ------------------------------------------------------------

df = pd.DataFrame(books)

# ------------------------------------------------------------
# Display Dataset Information
# ------------------------------------------------------------

print("\n===================================")
print("SCRAPING COMPLETED")
print("===================================")

print("\nTotal Books Scraped :", len(df))

print("\nColumns")
print(df.columns)

print("\nFirst Five Records")
print(df.head())

# ------------------------------------------------------------
# Save Raw Dataset
# ------------------------------------------------------------

df.to_csv("raw_books.csv", index=False)

print("\nRaw dataset saved as raw_books.csv")


Scraping Page 1...

Scraping Page 2...

Scraping Page 3...

Scraping Page 4...

Scraping Page 5...

SCRAPING COMPLETED

Total Books Scraped : 100

Columns
Index(['title', 'price', 'star_rating', 'availability', 'category'], dtype='object')

First Five Records
                                   title    price star_rating availability  \
0                   A Light in the Attic  Â£51.77       Three     In stock   
1                     Tipping the Velvet  Â£53.74         One     In stock   
2                             Soumission  Â£50.10         One     In stock   
3                          Sharp Objects  Â£47.82        Four     In stock   
4  Sapiens: A Brief History of Humankind  Â£54.23        Five     In stock   

             category  
0              Poetry  
1  Historical Fiction  
2             Fiction  
3             Mystery  
4             History  

Raw dataset saved as raw_books.csv


In [ ]:
# ============================================================
# DATA PIPELINE PROJECT
# PART 2 - DATA CLEANING AND TRANSFORMATION
# ============================================================

# ------------------------------------------------------------
# Display Basic Information
# ------------------------------------------------------------

print("=" * 60)
print("DATASET INFORMATION BEFORE CLEANING")
print("=" * 60)

print("\nFirst Five Records")
print(df.head())

print("\nDataset Shape")
print(df.shape)

print("\nColumn Names")
print(df.columns.tolist())

print("\nMissing Values")
print(df.isnull().sum())

# ------------------------------------------------------------
# Create a Copy of the Original DataFrame
# ------------------------------------------------------------

# This keeps the original data safe
clean_df = df.copy()

# ------------------------------------------------------------
# Convert Price from Text to Float
# ------------------------------------------------------------

print("\nCleaning Price Column...")

# Remove the £ symbol
clean_df["price_gbp"] = clean_df["price"].str.replace(
    "£",
    "",
    regex=False
)

# Convert to numeric
clean_df["price_gbp"] = pd.to_numeric(
    clean_df["price_gbp"],
    errors="coerce"
)

# ------------------------------------------------------------
# Convert Rating from Text to Integer
# ------------------------------------------------------------

print("Converting Ratings...")

rating_map = {

    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5

}

clean_df["rating"] = clean_df["star_rating"].map(
    rating_map
)

# ------------------------------------------------------------
# Convert Availability to Boolean
# ------------------------------------------------------------

print("Converting Availability...")

clean_df["in_stock"] = clean_df["availability"].str.contains(
    "In stock",
    case=False,
    na=False
)

# ------------------------------------------------------------
# Handle Missing Values
# ------------------------------------------------------------

print("Handling Missing Values...")

# Replace missing prices with median
price_median = clean_df["price_gbp"].median()

clean_df["price_gbp"] = clean_df["price_gbp"].fillna(
    price_median
)

# Replace missing ratings with median
rating_median = int(clean_df["rating"].median())

clean_df["rating"] = clean_df["rating"].fillna(
    rating_median
)

# Remove rows having missing title
clean_df = clean_df.dropna(
    subset=["title"]
)

# Remove rows having missing category
clean_df = clean_df.dropna(
    subset=["category"]
)

# ------------------------------------------------------------
# Convert GBP to INR
# ------------------------------------------------------------

print("Converting GBP to INR...")

GBP_TO_INR = 105.50

clean_df["price_inr"] = (
    clean_df["price_gbp"] * GBP_TO_INR
).round(2)

# ------------------------------------------------------------
# Arrange Columns
# ------------------------------------------------------------

clean_df = clean_df[
    [

        "title",

        "category",

        "price",

        "price_gbp",

        "price_inr",

        "star_rating",

        "rating",

        "availability",

        "in_stock"

    ]
]

# ------------------------------------------------------------
# Verify Data Types
# ------------------------------------------------------------

print("\n")
print("=" * 60)
print("DATA TYPES")
print("=" * 60)

print(clean_df.dtypes)

# ------------------------------------------------------------
# Check Missing Values Again
# ------------------------------------------------------------

print("\n")
print("=" * 60)
print("MISSING VALUES AFTER CLEANING")
print("=" * 60)

print(clean_df.isnull().sum())

# ------------------------------------------------------------
# Display Cleaned Data
# ------------------------------------------------------------

print("\n")
print("=" * 60)
print("FIRST FIVE CLEANED RECORDS")
print("=" * 60)

print(clean_df.head())

# ------------------------------------------------------------
# Save Cleaned Dataset
# ------------------------------------------------------------

clean_df.to_csv(
    "clean_books.csv",
    index=False
)

print("\nCleaned dataset saved as clean_books.csv")

# ------------------------------------------------------------
# Replace Original DataFrame
# ------------------------------------------------------------

# The remaining parts of the project will use df
df = clean_df.copy()

print("\n")
print("=" * 60)
print("DATA CLEANING COMPLETED SUCCESSFULLY")
print("=" * 60)

print("Total Books :", len(df))

print("Columns Available")

print(df.columns.tolist())

DATASET INFORMATION BEFORE CLEANING

First Five Records
                                   title    price star_rating availability  \
0                   A Light in the Attic  Â£51.77       Three     In stock   
1                     Tipping the Velvet  Â£53.74         One     In stock   
2                             Soumission  Â£50.10         One     In stock   
3                          Sharp Objects  Â£47.82        Four     In stock   
4  Sapiens: A Brief History of Humankind  Â£54.23        Five     In stock   

             category  
0              Poetry  
1  Historical Fiction  
2             Fiction  
3             Mystery  
4             History  

Dataset Shape
(100, 5)

Column Names
['title', 'price', 'star_rating', 'availability', 'category']

Missing Values
title           0
price           0
star_rating     0
availability    0
category        0
dtype: int64

Cleaning Price Column...
Converting Ratings...
Converting Availability...
Handling Missing Values...
Converting

In [ ]:
# ============================================================
# DATA PIPELINE PROJECT
# PART 3 - CREATE SQLITE DATABASE
# ============================================================

# ------------------------------------------------------------
# Import Required Library
# ------------------------------------------------------------

import sqlite3

# ------------------------------------------------------------
# Create SQLite Database
# ------------------------------------------------------------

print("=" * 60)
print("CREATING SQLITE DATABASE")
print("=" * 60)

# Create/Open Database
connection = sqlite3.connect("books.db")

# Create Cursor
cursor = connection.cursor()

print("Database Connected Successfully")

# ------------------------------------------------------------
# Remove Existing Tables (if they exist)
# ------------------------------------------------------------

cursor.execute("DROP TABLE IF EXISTS books")
cursor.execute("DROP TABLE IF EXISTS categories")

connection.commit()

print("Old Tables Removed")

# ------------------------------------------------------------
# Create Categories Table
# ------------------------------------------------------------

cursor.execute("""

CREATE TABLE categories(

    category_id INTEGER PRIMARY KEY AUTOINCREMENT,

    category_name TEXT UNIQUE

)

""")

connection.commit()

print("Categories Table Created")

# ------------------------------------------------------------
# Create Books Table
# ------------------------------------------------------------

cursor.execute("""

CREATE TABLE books(

    book_id INTEGER PRIMARY KEY AUTOINCREMENT,

    title TEXT NOT NULL,

    price_gbp REAL,

    price_inr REAL,

    rating INTEGER,

    in_stock INTEGER,

    category_id INTEGER,

    FOREIGN KEY(category_id)
    REFERENCES categories(category_id)

)

""")

connection.commit()

print("Books Table Created")

# ------------------------------------------------------------
# Extract Unique Categories
# ------------------------------------------------------------

print("\nExtracting Categories...")

unique_categories = sorted(df["category"].unique())

print(unique_categories)

# ------------------------------------------------------------
# Insert Categories
# ------------------------------------------------------------

print("\nInserting Categories...")

for category in unique_categories:

    cursor.execute(

        """
        INSERT INTO categories(category_name)
        VALUES(?)
        """,

        (category,)

    )

connection.commit()

print("Categories Inserted Successfully")

# ------------------------------------------------------------
# Read Categories Table
# ------------------------------------------------------------

category_df = pd.read_sql(

    "SELECT * FROM categories",

    connection

)

print("\nCategories Table")

print(category_df)

# ------------------------------------------------------------
# Merge Category IDs into DataFrame
# ------------------------------------------------------------

print("\nMerging Category IDs...")

books_df = pd.merge(

    df,

    category_df,

    left_on="category",

    right_on="category_name",

    how="left"

)

print("Merge Completed")

print("\nColumns")

print(books_df.columns.tolist())

# ------------------------------------------------------------
# Check Missing Category IDs
# ------------------------------------------------------------

missing = books_df["category_id"].isnull().sum()

print("\nMissing Category IDs :", missing)

# ------------------------------------------------------------
# Insert Books
# ------------------------------------------------------------

print("\nInserting Books...")

for index, row in books_df.iterrows():

    cursor.execute(

        """

        INSERT INTO books(

            title,

            price_gbp,

            price_inr,

            rating,

            in_stock,

            category_id

        )

        VALUES(?,?,?,?,?,?)

        """,

        (

            row["title"],

            float(row["price_gbp"]),

            float(row["price_inr"]),

            int(row["rating"]),

            int(row["in_stock"]),

            int(row["category_id"])

        )

    )

connection.commit()

print("Books Inserted Successfully")

# ------------------------------------------------------------
# Verify Books Table
# ------------------------------------------------------------

print("\nFirst 10 Books")

books_table = pd.read_sql(

    """

    SELECT *

    FROM books

    LIMIT 10

    """,

    connection

)

print(books_table)

# ------------------------------------------------------------
# Count Total Books
# ------------------------------------------------------------

count_df = pd.read_sql(

    """

    SELECT COUNT(*) AS Total_Books

    FROM books

    """,

    connection

)

print("\nTotal Books Stored")

print(count_df)

# ------------------------------------------------------------
# Display Database Tables
# ------------------------------------------------------------

tables = pd.read_sql(

    """

    SELECT name

    FROM sqlite_master

    WHERE type='table'

    """,

    connection

)

print("\nDatabase Tables")

print(tables)

# ------------------------------------------------------------
# Verify Category Count
# ------------------------------------------------------------

category_count = pd.read_sql(

    """

    SELECT COUNT(*) AS Total_Categories

    FROM categories

    """,

    connection

)

print("\nTotal Categories")

print(category_count)

# ------------------------------------------------------------
# Final Message
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("DATABASE CREATED SUCCESSFULLY")
print("=" * 60)

print("\nDatabase Name : books.db")

print("Tables Created : categories, books")

print("Primary Key : category_id, book_id")

print("Foreign Key : books.category_id -> categories.category_id")

print("Ready for SQL Queries")

CREATING SQLITE DATABASE
Database Connected Successfully
Old Tables Removed
Categories Table Created
Books Table Created

Extracting Categories...
['Add a comment', 'Art', 'Business', 'Childrens', 'Contemporary', 'Default', 'Fantasy', 'Fiction', 'Food and Drink', 'Health', 'Historical Fiction', 'History', 'Horror', 'Music', 'Mystery', 'New Adult', 'Nonfiction', 'Philosophy', 'Poetry', 'Politics', 'Romance', 'Science', 'Science Fiction', 'Self Help', 'Sequential Art', 'Spirituality', 'Thriller', 'Travel', 'Young Adult']

Inserting Categories...
Categories Inserted Successfully

Categories Table
    category_id       category_name
0             1       Add a comment
1             2                 Art
2             3            Business
3             4           Childrens
4             5        Contemporary
5             6             Default
6             7             Fantasy
7             8             Fiction
8             9      Food and Drink
9            10              Health
10 

In [ ]:
# ============================================================
# DATA PIPELINE PROJECT
# PART 4 - SQL QUERIES
# ============================================================

print("=" * 70)
print("EXECUTING SQL QUERIES")
print("=" * 70)

# Dictionary to store query outputs
query_results = {}

# ------------------------------------------------------------
# QUERY 1
# SELECT + WHERE
# ------------------------------------------------------------

print("\nQUERY 1 : SELECT + WHERE")

query1 = """
SELECT
    title,
    rating,
    price_gbp
FROM books
WHERE rating = 5;
"""

result1 = pd.read_sql(query1, connection)

query_results["Query 1"] = result1

print(result1)

# ------------------------------------------------------------
# QUERY 2
# ORDER BY + LIMIT
# ------------------------------------------------------------

print("\nQUERY 2 : ORDER BY + LIMIT")

query2 = """
SELECT
    title,
    price_gbp,
    price_inr
FROM books
ORDER BY price_gbp DESC
LIMIT 10;
"""

result2 = pd.read_sql(query2, connection)

query_results["Query 2"] = result2

print(result2)

# ------------------------------------------------------------
# QUERY 3
# DISTINCT
# ------------------------------------------------------------

print("\nQUERY 3 : DISTINCT")

query3 = """
SELECT DISTINCT
    rating
FROM books
ORDER BY rating;
"""

result3 = pd.read_sql(query3, connection)

query_results["Query 3"] = result3

print(result3)

# ------------------------------------------------------------
# QUERY 4
# BETWEEN
# ------------------------------------------------------------

print("\nQUERY 4 : BETWEEN")

query4 = """
SELECT
    title,
    price_gbp,
    rating
FROM books
WHERE price_gbp BETWEEN 20 AND 40;
"""

result4 = pd.read_sql(query4, connection)

query_results["Query 4"] = result4

print(result4)

# ------------------------------------------------------------
# QUERY 5
# INNER JOIN
# ------------------------------------------------------------

print("\nQUERY 5 : INNER JOIN")

query5 = """
SELECT

    b.title,

    c.category_name,

    b.rating,

    b.price_gbp

FROM books b

INNER JOIN categories c

ON b.category_id = c.category_id

ORDER BY

    c.category_name,

    b.rating DESC;
"""

result5 = pd.read_sql(query5, connection)

query_results["Query 5"] = result5

print(result5)

# ------------------------------------------------------------
# QUERY 6
# GROUP BY (Extra Query)
# ------------------------------------------------------------

print("\nQUERY 6 : GROUP BY")

query6 = """
SELECT

    c.category_name,

    COUNT(*) AS Total_Books,

    ROUND(AVG(b.price_gbp),2) AS Average_Price

FROM books b

JOIN categories c

ON b.category_id = c.category_id

GROUP BY c.category_name

ORDER BY Total_Books DESC;
"""

result6 = pd.read_sql(query6, connection)

query_results["Query 6"] = result6

print(result6)

# ------------------------------------------------------------
# SAVE QUERY OUTPUTS
# ------------------------------------------------------------

print("\nSaving Query Outputs...")

with open("sql_query_outputs.txt", "w", encoding="utf-8") as file:

    file.write("=" * 70 + "\n")
    file.write("SQL QUERY OUTPUTS\n")
    file.write("=" * 70 + "\n\n")

    file.write("QUERY 1\n")
    file.write(query1 + "\n")
    file.write(result1.to_string(index=False))
    file.write("\n\n")

    file.write("QUERY 2\n")
    file.write(query2 + "\n")
    file.write(result2.to_string(index=False))
    file.write("\n\n")

    file.write("QUERY 3\n")
    file.write(query3 + "\n")
    file.write(result3.to_string(index=False))
    file.write("\n\n")

    file.write("QUERY 4\n")
    file.write(query4 + "\n")
    file.write(result4.to_string(index=False))
    file.write("\n\n")

    file.write("QUERY 5\n")
    file.write(query5 + "\n")
    file.write(result5.to_string(index=False))
    file.write("\n\n")

    file.write("QUERY 6\n")
    file.write(query6 + "\n")
    file.write(result6.to_string(index=False))

print("Query results saved as sql_query_outputs.txt")

# ------------------------------------------------------------
# DISPLAY SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ALL SQL QUERIES EXECUTED SUCCESSFULLY")
print("=" * 70)

print("\nQueries Covered")

print("✓ SELECT")
print("✓ WHERE")
print("✓ ORDER BY")
print("✓ LIMIT")
print("✓ DISTINCT")
print("✓ BETWEEN")
print("✓ JOIN")
print("✓ GROUP BY (Extra)")

print("\nOutput File Created")

print("sql_query_outputs.txt")

EXECUTING SQL QUERIES

QUERY 1 : SELECT + WHERE
                                                title  rating price_gbp
0               Sapiens: A Brief History of Humankind       5      None
1                                         Set Me Free       5      None
2   Scott Pilgrim's Precious Little Life (Scott Pi...       5      None
3                           Rip it Up and Start Again       5      None
4                          Chase Me (Paris Nights #2)       5      None
5                                          Black Dust       5      None
6   Worlds Elsewhere: Journeys Around Shakespeareâ...       5      None
7   The Four Agreements: A Practical Guide to Pers...       5      None
8                                   The Elephant Tree       5      None
9                                      Sophie's World       5      None
10                        Private Paris (Private #10)       5      None
11  #HigherSelfie: Wake Up Your Life. Free Your So...       5      None
12              

In [ ]:
# ============================================================
# DATA PIPELINE PROJECT
# PART 5 - PANDAS SQL & MERGE COMPARISON
# ============================================================

print("=" * 70)
print("PART 5 - PANDAS SQL & MERGE COMPARISON")
print("=" * 70)

# ------------------------------------------------------------
# Read Books Table into DataFrame
# ------------------------------------------------------------

print("\nReading Books Table...")

books_df = pd.read_sql(

    "SELECT * FROM books",

    connection

)

print("Books Table Loaded Successfully")

print("\nTotal Books :", len(books_df))

# ------------------------------------------------------------
# Read Categories Table into DataFrame
# ------------------------------------------------------------

print("\nReading Categories Table...")

categories_df = pd.read_sql(

    "SELECT * FROM categories",

    connection

)

print("Categories Table Loaded Successfully")

print("\nTotal Categories :", len(categories_df))

# ------------------------------------------------------------
# Read JOIN Result Using SQL
# ------------------------------------------------------------

print("\nExecuting SQL JOIN...")

sql_join_query = """

SELECT

    b.title,

    c.category_name,

    b.rating,

    b.price_gbp,

    b.price_inr,

    b.in_stock

FROM books b

JOIN categories c

ON b.category_id = c.category_id

ORDER BY

    c.category_name,

    b.rating DESC,

    b.title;

"""

sql_join_df = pd.read_sql(

    sql_join_query,

    connection

)

print("SQL JOIN Completed")

# ------------------------------------------------------------
# Perform Same JOIN Using Pandas
# ------------------------------------------------------------

print("\nPerforming Pandas Merge...")

merge_df = pd.merge(

    books_df,

    categories_df,

    on="category_id",

    how="inner"

)

# Keep only required columns

merge_df = merge_df[

    [

        "title",

        "category_name",

        "rating",

        "price_gbp",

        "price_inr",

        "in_stock"

    ]

]

# Sort exactly like SQL

merge_df = merge_df.sort_values(

    by=[

        "category_name",

        "rating",

        "title"

    ],

    ascending=[

        True,

        False,

        True

    ]

).reset_index(drop=True)

# SQL DataFrame reset

sql_join_df = sql_join_df.reset_index(drop=True)

print("Pandas Merge Completed")

# ------------------------------------------------------------
# Compare Both DataFrames
# ------------------------------------------------------------

print("\nComparing SQL JOIN with Pandas Merge...")

comparison = sql_join_df.equals(merge_df)

print("\nAre Both Outputs Equal?")

print(comparison)

# ------------------------------------------------------------
# Display Sample Output
# ------------------------------------------------------------

print("\n")
print("=" * 70)
print("SQL JOIN OUTPUT")
print("=" * 70)

print(sql_join_df.head(10))

print("\n")
print("=" * 70)
print("PANDAS MERGE OUTPUT")
print("=" * 70)

print(merge_df.head(10))

# ------------------------------------------------------------
# Save Outputs
# ------------------------------------------------------------

sql_join_df.to_csv(

    "sql_join_output.csv",

    index=False

)

merge_df.to_csv(

    "pandas_merge_output.csv",

    index=False

)

print("\nCSV Files Saved")

# ------------------------------------------------------------
# Create Comparison Report
# ------------------------------------------------------------

with open(

    "comparison_report.txt",

    "w",

    encoding="utf-8"

) as file:

    file.write("=" * 70 + "\n")

    file.write("SQL JOIN vs PANDAS MERGE COMPARISON\n")

    file.write("=" * 70 + "\n\n")

    file.write("Total Books : ")

    file.write(str(len(books_df)))

    file.write("\n")

    file.write("Total Categories : ")

    file.write(str(len(categories_df)))

    file.write("\n\n")

    file.write("Outputs Match : ")

    file.write(str(comparison))

    file.write("\n")

print("Comparison Report Saved")

# ------------------------------------------------------------
# Display Database Tables
# ------------------------------------------------------------

print("\nDatabase Tables")

print(pd.read_sql(

    "SELECT name FROM sqlite_master WHERE type='table'",

    connection

))

# ------------------------------------------------------------
# Close Database Connection
# ------------------------------------------------------------

connection.close()

print("\nDatabase Connection Closed Successfully")

# ------------------------------------------------------------
# Final Summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PROJECT COMPLETED SUCCESSFULLY")
print("=" * 70)

print("\nFiles Created")

print("✓ raw_books.csv")
print("✓ clean_books.csv")
print("✓ books.db")
print("✓ sql_query_outputs.txt")
print("✓ sql_join_output.csv")
print("✓ pandas_merge_output.csv")
print("✓ comparison_report.txt")

print("\nAssignment Requirements Completed")

print("✓ Web Scraping")
print("✓ Data Cleaning")
print("✓ GBP to INR Conversion")
print("✓ SQLite Database")
print("✓ SQL Queries")
print("✓ Primary Key / Foreign Key")
print("✓ pd.read_sql()")
print("✓ pd.merge()")
print("✓ Output Comparison")

PART 5 - PANDAS SQL & MERGE COMPARISON

Reading Books Table...
Books Table Loaded Successfully

Total Books : 100

Reading Categories Table...
Categories Table Loaded Successfully

Total Categories : 29

Executing SQL JOIN...
SQL JOIN Completed

Performing Pandas Merge...
Pandas Merge Completed

Comparing SQL JOIN with Pandas Merge...

Are Both Outputs Equal?
True


SQL JOIN OUTPUT
                                               title  category_name  rating  \
0  The Mindfulness and Acceptance Workbook for An...  Add a comment       4   
1                                On a Midnight Clear  Add a comment       3   
2                                     The Art Forger  Add a comment       3   
3  Judo: Seven Steps to Black Belt (an Introducto...  Add a comment       2   
4        The Torch Is Passed: A Harding Family Story  Add a comment       1   
5                                     Wall and Piece            Art       4   
6  The Dirty Little Secrets of Getting Your Dream...       Bus